# 2b. AA↔3Di translation accuracy

**Paper:** Per-residue translation accuracy (AA→3Di and 3Di→AA vs ground truth). Independent of `2a_remote_homology.ipynb`. Figures in `3_figures.ipynb`.

Homology search uses AA→3Di only. This notebook scores **both directions**.

| Task | Prediction | Ground truth |
|------|------------|--------------|
| AA→3Di | `work/aa2di_fasta/DB_*_aa2di.fasta` | `work/GT_fasta/DB_di.fasta` |
| 3Di→AA | `work/di2aa_fasta/DB_*_di2aa.fasta` | `work/GT_fasta/DB_aa.fasta` |

Metrics: micro / macro accuracy, exact-match fraction. Default `SKIP_LENGTH_MISMATCH = True`: length-mismatched sequences are counted but excluded from accuracy.

| Method | aa2di filename | di2aa filename |
|--------|----------------|----------------|
| ESM3-3Di | `DB_ESM3_aa2di.fasta` | `DB_ESM3_di2aa.fasta` |
| ESM3-LoRA | `DB_ESM3_LoRA_aa2di.fasta` | `DB_ESM3_LoRA_di2aa.fasta` |
| ProstT5 (translate) | `DB_ProstT5_translate_aa2di.fasta` | `DB_ProstT5_translate_di2aa.fasta` |
| SaProt | `DB_SaProt_aa2di.fasta` | `DB_SaProt_di2aa.fasta` |

| | Path |
|--|------|
| Output | `work/metrics/translation/{task}_{method}_per_seq.tsv` |
| | `work/metrics/translation/{aa2di,di2aa}_summary.csv`, `translation_summary.csv` |

**Node:** login is enough. Minutes.

**Next:** `3_figures.ipynb`.


## Environment


In [ ]:
NOTEBOOK_NAME = "2b_translation_accuracy.ipynb"

import os
import platform
import subprocess
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if not (cwd / NOTEBOOK_NAME).is_file():
    raise SystemExit(
        f"Start this notebook from the project root (cwd must contain {NOTEBOOK_NAME}). "
        f"Current cwd: {cwd}"
    )

CONDA_ENV = "ESM3_3Di_5090"
print("notebook:", NOTEBOOK_NAME)
print("cwd:", cwd)
print("python:", sys.executable)
print("version:", sys.version.split()[0])
print("platform:", platform.platform())
print("CONDA_DEFAULT_ENV:", os.environ.get("CONDA_DEFAULT_ENV", "(unset)"))

if CONDA_ENV not in sys.executable:
    expected = Path.home() / ".conda" / "envs" / CONDA_ENV / "bin" / "python"
    raise SystemExit(
        f"Kernel is not {CONDA_ENV} (current: {sys.executable}). "
        f"Select kernel {CONDA_ENV} and Restart. Do not pip into miniforge3 python3.12. "
        f"Expected: {expected}"
    )


def _bin_version(name: str) -> str:
    path = cwd / "bin" / name
    if not path.is_file():
        return "(not installed yet; run 0_prepare_scope40.ipynb)"
    try:
        proc = subprocess.run([str(path), "version"], capture_output=True, text=True, check=False)
        lines = (proc.stdout or proc.stderr or "").strip().splitlines()
        return lines[0] if lines else "(unknown)"
    except OSError as exc:
        return f"(failed: {exc})"


print("foldseek:", _bin_version("foldseek"))
print("mmseqs:", _bin_version("mmseqs"))


## Configuration

本格在五本 notebook 中**字节级相同**。改方法表、搜索参数或 URL 时：只改 `0_prepare_scope40.ipynb` 这一格，再整格复制到另外四本。发布前可用 checksum 核对五本是否一致。


In [ ]:
# =============================================================================
# Configuration — copy this entire cell into all five notebooks.
# Change methods or search parameters here in 0_prepare_scope40.ipynb, then
# paste the same cell into 1_build / 2a / 2b / 3_figures.
# =============================================================================
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
HOME = ROOT.parent

CONDA_ENV = "ESM3_3Di_5090"
FOLDSEEK_VERSION = "10-941cd33"
MMSEQS_VERSION = "18-8cc5c"

TEMP = ROOT / "tmp"
WORK_DIR = ROOT / "work"
BIN_DIR = ROOT / "bin"
WORK_TMP_DIR = WORK_DIR / "tmp"

GT_FASTA_DIR = WORK_DIR / "GT_fasta"
AA_FASTA = GT_FASTA_DIR / "DB_aa.fasta"
GT_DI_FASTA = GT_FASTA_DIR / "DB_di.fasta"

AA2DI_FASTA_DIR = WORK_DIR / "aa2di_fasta"
DI2AA_FASTA_DIR = WORK_DIR / "di2aa_fasta"

DBS_DIR = WORK_DIR / "DB"
FOLDSEEK_GT_DIR = DBS_DIR / "foldseek_DB"
MMSEQS_GT_DIR = DBS_DIR / "mmseqs_DB"

LABEL_DIR = WORK_DIR / "labels"
SCOP_LOOKUP = LABEL_DIR / "scop_lookup.tsv"
LEGACY_LABEL_DIR = WORK_DIR / "lable"

ALN_DIR = WORK_DIR / "aln"
METRICS_DIR = WORK_DIR / "metrics"
FIGURES_DIR = WORK_DIR / "figures"
TRANSLATION_METRICS_DIR = METRICS_DIR / "translation"
WORK_BUNDLE = WORK_DIR / "scope40_work_bundle.tar.gz"

FOLDSEEK_BIN = BIN_DIR / "foldseek"
MMSEQS_BIN = BIN_DIR / "mmseqs"

FOLDSEEK_URL = (
    "https://github.com/steineggerlab/foldseek/releases/download/"
    f"{FOLDSEEK_VERSION}/foldseek-linux-avx2.tar.gz"
)
MMSEQS_URL = (
    "https://github.com/soedinglab/MMseqs2/releases/download/"
    f"{MMSEQS_VERSION}/mmseqs-linux-avx2.tar.gz"
)
FOLDSEEK_TMP_DIR = TEMP / "foldseek"
MMSEQS_TMP_DIR = TEMP / "mmseqs"
FOLDSEEK_TARBALL = TEMP / "foldseek-linux-avx2.tar.gz"
MMSEQS_TARBALL = TEMP / "mmseqs-linux-avx2.tar.gz"

SCOP_CLA_NAME = "dir.cla.scope.2.08-stable.txt"
SCOP_DES_NAME = "dir.des.scope.2.08-stable.txt"
SOURCE_ARCHIVE_NAME = "pdbstyle-sel-gs-bib-40-2.08.tgz"
SCOP_CLA_FALLBACK = HOME / "SCOPE" / SCOP_CLA_NAME
HF_BASE = "https://huggingface.co/datasets/caijihuize/scope40_pdbstyle/resolve/main"

# Foldseek / predicted 3Di: aligned with new_scope40 easy-search
EASY_SEARCH_PARAMS = {
    "sensitivity": 9.5,
    "max_seqs": 2000,
    "evalue": 10.0,
    "threads": 64,
}
# MMseqs2: aligned with foldseek-analysis/scopbenchmark/scripts/runMMseqs.sh
MMSEQS_SEARCH_PARAMS = {
    "sensitivity": 7.5,
    "max_seqs": 2000,
    "evalue": 10000,
    "threads": 64,
    "add_backtrace": True,
}
PREPARE_THREADS = 16
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# Homology search methods. protocol must not be mixed as one AUC.
METHODS: list[dict] = [
    {"name": "Foldseek (AA+3Di)", "key": "foldseek", "engine": "foldseek", "aa2di": None, "protocol": "hitlist"},
    {"name": "MMseqs2", "key": "mmseqs", "engine": "mmseqs", "aa2di": None, "protocol": "catalog"},
    {"name": "ESM3-3Di", "key": "ESM3", "engine": "foldseek", "aa2di": "DB_ESM3_aa2di.fasta", "protocol": "hitlist"},
    {"name": "ESM3-LoRA", "key": "ESM3_LoRA", "engine": "foldseek", "aa2di": "DB_ESM3_LoRA_aa2di.fasta", "protocol": "hitlist"},
    {"name": "ProstT5 (translate)", "key": "ProstT5", "engine": "foldseek", "aa2di": "DB_ProstT5_translate_aa2di.fasta", "protocol": "hitlist"},
    {"name": "SaProt", "key": "SaProt", "engine": "foldseek", "aa2di": "DB_SaProt_aa2di.fasta", "protocol": "hitlist"},
]
# Translation accuracy (bidirectional). Homology search uses aa2di only.
TRANSLATION_METHODS: list[dict] = [
    {"name": "ESM3-3Di", "key": "ESM3", "aa2di": "DB_ESM3_aa2di.fasta", "di2aa": "DB_ESM3_di2aa.fasta"},
    {"name": "ESM3-LoRA", "key": "ESM3_LoRA", "aa2di": "DB_ESM3_LoRA_aa2di.fasta", "di2aa": "DB_ESM3_LoRA_di2aa.fasta"},
    {"name": "ProstT5 (translate)", "key": "ProstT5", "aa2di": "DB_ProstT5_translate_aa2di.fasta", "di2aa": "DB_ProstT5_translate_di2aa.fasta"},
    {"name": "SaProt", "key": "SaProt", "aa2di": "DB_SaProt_aa2di.fasta", "di2aa": "DB_SaProt_di2aa.fasta"},
]
PALETTE = {
    "Foldseek (AA+3Di)": "#2b5c8f",
    "MMseqs2": "#666666",
    "ESM3-3Di": "#d95f02",
    "ESM3-LoRA": "#1b9e77",
    "ProstT5 (translate)": "#7570b3",
    "SaProt": "#e7298a",
}


def run_cmd(argv: list[str]) -> None:
    """Print then run an external command. Logs are supplementary material."""
    print("[CMD]", " ".join(str(x) for x in argv), flush=True)
    subprocess.run([str(x) for x in argv], check=True)


def require_file(path: Path, hint: str) -> Path:
    """Fail with a pointer to the upstream notebook if a required file is missing."""
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}\n{hint}")
    return path


def meta_path(output: Path) -> Path:
    return output.with_name(output.name + ".meta.json")


def skip_if_exists(output: Path, payload: dict | None = None, skip_existing: bool = True) -> bool:
    """Skip when output exists. If payload is given, require a matching .meta.json.

    MMseqs TSV without a fingerprint is treated as stale (catalog protocol change).
    Other outputs without a fingerprint are kept; set SKIP_EXISTING=False to force.
    """
    if not skip_existing:
        return False
    if not output.is_file() or output.stat().st_size == 0:
        return False
    if payload is None:
        return True
    meta = meta_path(output)
    if not meta.is_file():
        if payload.get("engine") == "mmseqs":
            print(f"[rerun] {output.name}: no fingerprint; MMseqs catalog params need a fresh search")
            return False
        print(f"[warn] {output.name}: no fingerprint; keeping existing file (set SKIP_EXISTING=False to re-run)")
        return True
    try:
        stored = json.loads(meta.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return False
    if stored != payload:
        print(f"[rerun] {output.name}: Configuration changed")
        return False
    return True


def write_meta(output: Path, payload: dict) -> None:
    meta_path(output).write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def search_params_payload(engine: str, threads: int | None = None) -> dict:
    params = dict(MMSEQS_SEARCH_PARAMS if engine == "mmseqs" else EASY_SEARCH_PARAMS)
    params["engine"] = engine
    if threads is not None:
        params["threads"] = int(threads)
    return params


def method_by_key(method_key: str) -> dict:
    for row in METHODS:
        if row["key"] == method_key:
            return row
    raise KeyError(f"Unknown method_key: {method_key}")


def predicted_methods() -> list[dict]:
    return [row for row in METHODS if row["aa2di"] is not None]


def db_prefix(method_key: str) -> Path:
    return DBS_DIR / f"{method_key}_DB" / "DB"


def aln_tsv(method_key: str) -> Path:
    return ALN_DIR / f"{method_key}_easy.tsv"


def aln_tmp_dir(method_key: str) -> Path:
    return WORK_TMP_DIR / f"easy_{method_key}"


def metric_prefix(method_key: str) -> Path:
    return METRICS_DIR / f"{method_key}_easy"


def translation_per_seq_path(task: str, method_key: str) -> Path:
    return TRANSLATION_METRICS_DIR / f"{task}_{method_key}_per_seq.tsv"


def translation_summary_path(task: str) -> Path:
    return TRANSLATION_METRICS_DIR / f"{task}_summary.csv"


def scop_cla_path() -> Path:
    for path in (TEMP / SCOP_CLA_NAME, SCOP_CLA_FALLBACK):
        if path.is_file():
            return path
    return TEMP / SCOP_CLA_NAME


def work_ready() -> bool:
    return (
        (FOLDSEEK_GT_DIR / "DB").is_file()
        and (MMSEQS_GT_DIR / "DB").is_file()
        and AA_FASTA.is_file()
        and GT_DI_FASTA.is_file()
        and SCOP_LOOKUP.is_file()
    )


def ensure_work_dirs() -> None:
    for directory in (
        TEMP, BIN_DIR, GT_FASTA_DIR, AA2DI_FASTA_DIR, DI2AA_FASTA_DIR, DBS_DIR,
        LABEL_DIR, ALN_DIR, METRICS_DIR, TRANSLATION_METRICS_DIR, FIGURES_DIR, WORK_TMP_DIR,
    ):
        directory.mkdir(parents=True, exist_ok=True)
    legacy = LEGACY_LABEL_DIR / "scop_lookup.tsv"
    if not SCOP_LOOKUP.is_file() and legacy.is_file():
        shutil.copy2(legacy, SCOP_LOOKUP)
        print(f"[ok] migrated {legacy} -> {SCOP_LOOKUP}")


def cleanup_tmp(*, also_work_tmp: bool = True) -> None:
    """Remove tmp/ and work/tmp/ only. Keep work/ products and bin/."""
    targets = [TEMP]
    if also_work_tmp:
        targets.append(WORK_TMP_DIR)
    for path in targets:
        if path.exists():
            shutil.rmtree(path)
            print(f"[ok] cleaned {path}")
        else:
            print(f"[skip] {path} (absent)")


print("ROOT:", ROOT)
print("conda env:", CONDA_ENV)
print("Foldseek:", FOLDSEEK_VERSION, "MMseqs:", MMSEQS_VERSION)
print("METHODS:", [(m["key"], m["engine"], m["protocol"]) for m in METHODS])


## Run flags


In [ ]:
SKIP_EXISTING = True
ONLY_METHODS = None  # e.g. ["ESM3", "ProstT5"]; None = all TRANSLATION_METHODS
SKIP_LENGTH_MISMATCH = True  # True = drop length-mismatched pairs from accuracy

ensure_work_dirs()
print("GT aa:", AA_FASTA)
print("GT di:", GT_DI_FASTA)
print("out:", TRANSLATION_METRICS_DIR)
print(f"SKIP_EXISTING={SKIP_EXISTING}  SKIP_LENGTH_MISMATCH={SKIP_LENGTH_MISMATCH}")
print("ONLY_METHODS:", ONLY_METHODS)


## Helpers


In [ ]:
from dataclasses import dataclass

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


def load_fasta(path: Path) -> dict[str, str]:
    """Minimal FASTA reader — Biopython is not required for this notebook."""
    out: dict[str, str] = {}
    name: str | None = None
    chunks: list[str] = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.rstrip("\n")
            if not line:
                continue
            if line.startswith(">"):
                if name is not None:
                    out[name] = "".join(chunks)
                name = line[1:].split()[0]
                chunks = []
            else:
                chunks.append(line.strip())
        if name is not None:
            out[name] = "".join(chunks)
    return out


@dataclass
class TaskSummary:
    task: str
    method_key: str
    label: str
    n_seqs: int
    n_residues: int
    micro_acc: float
    macro_acc: float
    exact_match_frac: float
    length_mismatch_n: int
    missing_in_pred_n: int
    invalid_char_frac: float


def _compare_sequences(
    pred: str,
    gt: str,
    *,
    uppercase: bool,
    valid_chars: set[str] | None,
) -> tuple[float, int, int, float]:
    if uppercase:
        pred, gt = pred.upper(), gt.upper()
    if len(pred) != len(gt):
        raise ValueError("length mismatch")
    if len(pred) == 0:
        return 1.0, 1, 0, 0.0
    correct = sum(p == g for p, g in zip(pred, gt, strict=True))
    acc = correct / len(pred)
    exact = int(pred == gt)
    invalid = 0
    if valid_chars is not None:
        invalid = sum(1 for p in pred if p not in valid_chars)
    return acc, exact, len(pred) - correct, invalid / len(pred)


print("helpers ready")


## Step 1 — Score each method against ground-truth FASTA

Missing prediction files are skipped (so 2b can run with a subset). Length mismatches follow `SKIP_LENGTH_MISMATCH`.


In [ ]:
def evaluate_task(
    task: str,
    method_key: str,
    label: str,
    pred_path: Path,
    gt_path: Path,
    *,
    uppercase: bool,
    valid_chars: set[str] | None,
    skip_existing: bool = True,
) -> TaskSummary | None:
    per_seq_out = translation_per_seq_path(task, method_key)
    if skip_if_exists(per_seq_out, payload=None, skip_existing=skip_existing):
        print(f"[skip] {per_seq_out}")
        row = pd.read_csv(per_seq_out, sep="\t")
        if row.empty:
            return None
        return TaskSummary(
            task=task,
            method_key=method_key,
            label=label,
            n_seqs=len(row),
            n_residues=int(row["length"].sum()),
            micro_acc=float(row["n_match"].sum() / row["length"].sum()) if row["length"].sum() else 0.0,
            macro_acc=float(row["acc"].mean()),
            exact_match_frac=float(row["exact_match"].mean()),
            length_mismatch_n=0,
            missing_in_pred_n=0,
            invalid_char_frac=float(row["invalid_char_frac"].mean()) if "invalid_char_frac" in row else 0.0,
        )

    if not pred_path.is_file():
        print(f"[skip] no prediction file: {pred_path}")
        return None
    require_file(gt_path, hint="Run 0_prepare_scope40.ipynb.")

    pred = load_fasta(pred_path)
    gt = load_fasta(gt_path)
    gt_ids = set(gt)
    pred_ids = set(pred)
    missing = gt_ids - pred_ids
    extra = pred_ids - gt_ids
    if extra:
        print(f"[warn] {label} {task}: {len(extra)} extra predicted ids (ignored)")
    if missing:
        print(f"[warn] {label} {task}: {len(missing)} GT ids have no prediction")

    rows: list[dict] = []
    length_mismatch = 0
    total_correct = 0
    total_len = 0
    invalid_fracs: list[float] = []

    for qid in sorted(gt_ids & pred_ids):
        p, g = pred[qid], gt[qid]
        if len(p) != len(g):
            length_mismatch += 1
            if SKIP_LENGTH_MISMATCH:
                continue
            n = min(len(p), len(g))
            p, g = p[:n], g[:n]
        n = len(g)
        acc, exact, n_mis, inv_frac = _compare_sequences(
            p, g, uppercase=uppercase, valid_chars=valid_chars,
        )
        n_match = n - n_mis
        rows.append({
            "qid": qid,
            "length": n,
            "acc": acc,
            "exact_match": exact,
            "n_mismatch": n_mis,
            "n_match": n_match,
            "invalid_char_frac": inv_frac,
        })
        total_correct += n_match
        total_len += n
        invalid_fracs.append(inv_frac)

    if not rows:
        print(f"[missing] {label} {task}: no comparable sequences")
        return None

    df = pd.DataFrame(rows)
    per_seq_out.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(per_seq_out, sep="\t", index=False)
    print(f"[ok] {per_seq_out}  n={len(df)}  length_mismatch={length_mismatch}")
    return TaskSummary(
        task=task,
        method_key=method_key,
        label=label,
        n_seqs=len(df),
        n_residues=total_len,
        micro_acc=total_correct / total_len if total_len else 0.0,
        macro_acc=float(df["acc"].mean()),
        exact_match_frac=float(df["exact_match"].mean()),
        length_mismatch_n=length_mismatch,
        missing_in_pred_n=len(missing),
        invalid_char_frac=float(sum(invalid_fracs) / len(invalid_fracs)) if invalid_fracs else 0.0,
    )


def run_all(skip_existing: bool = True) -> pd.DataFrame:
    summaries: list[TaskSummary] = []
    methods = TRANSLATION_METHODS
    if ONLY_METHODS is not None:
        methods = [m for m in methods if m["key"] in ONLY_METHODS]

    gt_di_chars: set[str] | None = None
    if GT_DI_FASTA.is_file():
        gt_di_chars = set("".join(load_fasta(GT_DI_FASTA).values()).upper())

    for method in methods:
        label, key = method["name"], method["key"]
        print(f"\n=== {label} ===")
        s_aa2di = evaluate_task(
            "aa2di", key, label,
            AA2DI_FASTA_DIR / method["aa2di"], GT_DI_FASTA,
            uppercase=True, valid_chars=gt_di_chars, skip_existing=skip_existing,
        )
        if s_aa2di is not None:
            summaries.append(s_aa2di)
        s_di2aa = evaluate_task(
            "di2aa", key, label,
            DI2AA_FASTA_DIR / method["di2aa"], AA_FASTA,
            uppercase=False, valid_chars=STANDARD_AA, skip_existing=skip_existing,
        )
        if s_di2aa is not None:
            summaries.append(s_di2aa)

    if not summaries:
        return pd.DataFrame()

    df = pd.DataFrame([s.__dict__ for s in summaries])
    for task in ("aa2di", "di2aa"):
        sub = df[df["task"] == task]
        if not sub.empty:
            path = translation_summary_path(task)
            sub.to_csv(path, index=False)
            print(f"\n[ok] {path}")
    combined = TRANSLATION_METRICS_DIR / "translation_summary.csv"
    df.to_csv(combined, index=False)
    print(f"[ok] {combined}")
    return df


require_file(AA_FASTA, hint="Run 0_prepare_scope40.ipynb.")
require_file(GT_DI_FASTA, hint="Run 0_prepare_scope40.ipynb.")
summary_df = run_all(skip_existing=SKIP_EXISTING)
display(summary_df)
print("Next: 3_figures.ipynb")


## Verify


In [ ]:
combined = TRANSLATION_METRICS_DIR / "translation_summary.csv"
require_file(combined, hint="Step 1 produced no summary (missing all prediction FASTA?).")
print(combined)
display(pd.read_csv(combined)[["task", "label", "n_seqs", "micro_acc", "macro_acc", "exact_match_frac", "length_mismatch_n"]])
print("[ok] translation accuracy complete")


## Cleanup


In [ ]:
cleanup_tmp(also_work_tmp=True)
print("Cleanup done. Products remain under work/ and bin/.")
